#Imports

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import joblib

#Load Data

In [18]:
df = pd.read_csv("Loan_default.csv")
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


#Basic Checks

In [19]:
df.info()
df.isnull().sum()
df['Default'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   LoanID          255347 non-null  object 
 1   Age             255347 non-null  int64  
 2   Income          255347 non-null  int64  
 3   LoanAmount      255347 non-null  int64  
 4   CreditScore     255347 non-null  int64  
 5   MonthsEmployed  255347 non-null  int64  
 6   NumCreditLines  255347 non-null  int64  
 7   InterestRate    255347 non-null  float64
 8   LoanTerm        255347 non-null  int64  
 9   DTIRatio        255347 non-null  float64
 10  Education       255347 non-null  object 
 11  EmploymentType  255347 non-null  object 
 12  MaritalStatus   255347 non-null  object 
 13  HasMortgage     255347 non-null  object 
 14  HasDependents   255347 non-null  object 
 15  LoanPurpose     255347 non-null  object 
 16  HasCoSigner     255347 non-null  object 
 17  Default   

,count
Default,
0,225694
1,29653


#Encode Categorical

In [20]:
df = df.drop(columns=["LoanID"])

df = pd.get_dummies(df, drop_first=True)

#Split

In [21]:
X = df.drop("Default", axis=1)
y = df["Default"]

#Saved Coulmns

In [22]:
import joblib
joblib.dump(X.columns.tolist(), "columns.pkl")

['columns.pkl']

#Test-Train Spilt

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Models

## 1. Logistic Regression
(For probability)

In [24]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000)

## 2. Random Forest
(for classification)

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=150,
                       random_state=42)

# Evaluate

In [26]:
from sklearn.metrics import accuracy_score, classification_report

print("RF Accuracy:", accuracy_score(y_test, rf_model.predict(X_test)))
print(classification_report(y_test, rf_model.predict(X_test)))

RF Accuracy: 0.7284119835519874
              precision    recall  f1-score   support

           0       0.94      0.74      0.83     45170
           1       0.24      0.63      0.35      5900

    accuracy                           0.73     51070
   macro avg       0.59      0.68      0.59     51070
weighted avg       0.86      0.73      0.77     51070



# Debugging

In [27]:
print("RF Probabilities:")
print(rf_model.predict_proba(X_test[:10])[:,1])

print("LR Probabilities:")
print(lr_model.predict_proba(X_test[:10])[:,1])

RF Probabilities:
[0.29740252 0.29915154 0.46796721 0.43868639 0.47658994 0.57625379
 0.3024268  0.31839303 0.26822528 0.47392102]
LR Probabilities:
[0.29089401 0.26873933 0.4826799  0.3197006  0.5364913  0.65110055
 0.36135105 0.29392717 0.16862071 0.50212887]


# Models Saved

In [28]:
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(lr_model, "lr_model.pkl")

['lr_model.pkl']

# Saving Features

In [34]:
joblib.dump(X.columns.tolist(), "columns.pkl")

['columns.pkl']